In [8]:
import pandas as pd
import numpy as np

In [5]:
df_illinois = pd.read_parquet('IllinoisViolentCrime.parquet')
print(f"Dataframe shape before dropping any NaN value: {df_illinois.shape}")
df_illinois= df_illinois.dropna(axis=1)
print(f"Dataframe shape after dropping NaN values: {df_illinois.shape}")
df_illinois.head(5)

Dataframe shape before dropping any NaN value: (1272576, 14)
Dataframe shape after dropping NaN values: (1272576, 14)


,OFFENSE_ID,INCIDENT_ID,OFFENSE_CODE,OFFENSE_NAME,OFFENSE_CATEGORY_NAME,AGENCY_ID,INCIDENT_DATE,INCIDENT_HOUR,INCIDENT_DATETIME,UCR_AGENCY_NAME,NCIC_AGENCY_NAME,PUB_AGENCY_NAME,COUNTY_NAME,CRIME_GROUP
0,172305650,143439540,13B,Simple Assault,Assault Offenses,4677,2021-01-24,1.0,2021-01-24 01:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,ASSAULT
1,172305651,143439541,13B,Simple Assault,Assault Offenses,4677,2021-02-26,15.0,2021-02-26 15:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,ASSAULT
2,172305652,143439542,11A,Rape,Sex Offenses,4677,2021-02-27,14.0,2021-02-27 14:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,ASSAULT
3,172305654,143439544,13B,Simple Assault,Assault Offenses,4677,2021-02-27,22.0,2021-02-27 22:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,ASSAULT
4,172305655,143439545,23F,Theft From Motor Vehicle,Larceny/Theft Offenses,4677,2021-02-28,2.0,2021-02-28 02:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,THEFT


In [21]:
df_illinois = df_illinois.rename(columns={'INCIDENT_DATE':'Date'})
df_illinois

,OFFENSE_ID,INCIDENT_ID,OFFENSE_CODE,OFFENSE_NAME,OFFENSE_CATEGORY_NAME,AGENCY_ID,Date,INCIDENT_HOUR,INCIDENT_DATETIME,UCR_AGENCY_NAME,NCIC_AGENCY_NAME,PUB_AGENCY_NAME,COUNTY_NAME,CRIME_GROUP
0,172305650,143439540,13B,Simple Assault,Assault Offenses,4677,2021-01-24,1.0,2021-01-24 01:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,ASSAULT
1,172305651,143439541,13B,Simple Assault,Assault Offenses,4677,2021-02-26,15.0,2021-02-26 15:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,ASSAULT
2,172305652,143439542,11A,Rape,Sex Offenses,4677,2021-02-27,14.0,2021-02-27 14:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,ASSAULT
3,172305654,143439544,13B,Simple Assault,Assault Offenses,4677,2021-02-27,22.0,2021-02-27 22:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,ASSAULT
4,172305655,143439545,23F,Theft From Motor Vehicle,Larceny/Theft Offenses,4677,2021-02-28,2.0,2021-02-28 02:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,THEFT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1937363,246599225,208138576,23C,Shoplifting,Larceny/Theft Offenses,5684,2024-09-30,12.0,2024-09-30 12:00:00,CHICAGO,CHICAGO PD,Chicago,COOK,THEFT
1937364,246599226,208138577,23H,All Other Larceny,Larceny/Theft Offenses,5684,2024-11-20,13.0,2024-11-20 13:00:00,CHICAGO,CHICAGO PD,Chicago,COOK,THEFT
1937366,246599227,208138578,23H,All Other Larceny,Larceny/Theft Offenses,5684,2024-03-01,9.0,2024-03-01 09:00:00,CHICAGO,CHICAGO PD,Chicago,COOK,THEFT
1937367,246599228,208138579,23H,All Other Larceny,Larceny/Theft Offenses,5684,2024-03-08,10.0,2024-03-08 10:00:00,CHICAGO,CHICAGO PD,Chicago,COOK,THEFT


# Shift & Target

In [ ]:

df_shift = df_illinois.copy()
df_shift["Date"] = pd.to_datetime(df_shift["Date"], errors="coerce")


In [23]:

# 3) Assign each record to a shift and shift_date (anchored to shift start date)
# hour = df_shift["Date"].dt.hour
hour = df_shift['INCIDENT_HOUR']
df_shift["shift"] = np.select(
    [hour.between(6, 13), hour.between(14, 21)],
    ["morning_noon", "afternoon_night"],
    default="overnight"
)
df_shift["shift_date"] = df_shift["Date"].dt.floor("D")
df_shift.loc[hour < 6, "shift_date"] -= pd.Timedelta(days=1)


In [24]:
df_shift

,OFFENSE_ID,INCIDENT_ID,OFFENSE_CODE,OFFENSE_NAME,OFFENSE_CATEGORY_NAME,AGENCY_ID,Date,INCIDENT_HOUR,INCIDENT_DATETIME,UCR_AGENCY_NAME,NCIC_AGENCY_NAME,PUB_AGENCY_NAME,COUNTY_NAME,CRIME_GROUP,shift,shift_date
0,172305650,143439540,13B,Simple Assault,Assault Offenses,4677,2021-01-24,1.0,2021-01-24 01:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,ASSAULT,overnight,2021-01-23
1,172305651,143439541,13B,Simple Assault,Assault Offenses,4677,2021-02-26,15.0,2021-02-26 15:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,ASSAULT,afternoon_night,2021-02-26
2,172305652,143439542,11A,Rape,Sex Offenses,4677,2021-02-27,14.0,2021-02-27 14:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,ASSAULT,afternoon_night,2021-02-27
3,172305654,143439544,13B,Simple Assault,Assault Offenses,4677,2021-02-27,22.0,2021-02-27 22:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,ASSAULT,overnight,2021-02-27
4,172305655,143439545,23F,Theft From Motor Vehicle,Larceny/Theft Offenses,4677,2021-02-28,2.0,2021-02-28 02:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,THEFT,overnight,2021-02-27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1937363,246599225,208138576,23C,Shoplifting,Larceny/Theft Offenses,5684,2024-09-30,12.0,2024-09-30 12:00:00,CHICAGO,CHICAGO PD,Chicago,COOK,THEFT,morning_noon,2024-09-30
1937364,246599226,208138577,23H,All Other Larceny,Larceny/Theft Offenses,5684,2024-11-20,13.0,2024-11-20 13:00:00,CHICAGO,CHICAGO PD,Chicago,COOK,THEFT,morning_noon,2024-11-20
1937366,246599227,208138578,23H,All Other Larceny,Larceny/Theft Offenses,5684,2024-03-01,9.0,2024-03-01 09:00:00,CHICAGO,CHICAGO PD,Chicago,COOK,THEFT,morning_noon,2024-03-01
1937367,246599228,208138579,23H,All Other Larceny,Larceny/Theft Offenses,5684,2024-03-08,10.0,2024-03-08 10:00:00,CHICAGO,CHICAGO PD,Chicago,COOK,THEFT,morning_noon,2024-03-08


In [ ]:
# 1) Aggregate crimes per UCR Agency Name per shift, then build master grid (every agency × date × shift)
tile_shift_counts = (
    df_shift.groupby(["UCR_AGENCY_NAME", "shift_date", "shift"]).size().reset_index(name="crime_count")
)
shift_order = ["morning_noon", "afternoon_night", "overnight"]
index = pd.MultiIndex.from_product(
    [df_shift["UCR_AGENCY_NAME"].unique(),
     pd.date_range(df_shift["shift_date"].min(), df_shift["shift_date"].max(), freq="D"),
     shift_order],
    names=["UCR_AGENCY_NAME", "shift_date", "shift"]
)
master_grid = pd.DataFrame(index=index).reset_index()


In [ ]:
# 2) Merge actual counts, fill zeros, create binary target
final_df = pd.merge(master_grid, tile_shift_counts, on=["UCR_AGENCY_NAME", "shift_date", "shift"], how="left")
final_df["crime_count"] = final_df["crime_count"].fillna(0)
final_df["target"] = (final_df["crime_count"] > 0).astype(int)
final_df


,UCR_AGENCY_NAME,shift_date,shift,crime_count,target
0,QUINCY,2020-12-31,morning_noon,0.0,0
1,QUINCY,2020-12-31,afternoon_night,0.0,0
2,QUINCY,2020-12-31,overnight,0.0,0
3,QUINCY,2021-01-01,morning_noon,0.0,0
4,QUINCY,2021-01-01,afternoon_night,0.0,0
...,...,...,...,...,...
2719315,JOHN A. LOGAN COLLEGE,2024-12-30,afternoon_night,0.0,0
2719316,JOHN A. LOGAN COLLEGE,2024-12-30,overnight,0.0,0
2719317,JOHN A. LOGAN COLLEGE,2024-12-31,morning_noon,0.0,0
2719318,JOHN A. LOGAN COLLEGE,2024-12-31,afternoon_night,0.0,0


In [ ]:
# 3) Add Date and hour columns
# hour reflects the first (earliest) crime in that specific tile and shift;
# falls back to shift start hour for zero-crime rows
shift_start_hour = {"morning_noon": 6, "afternoon_night": 14, "overnight": 22}
final_df["Date"] = final_df["shift_date"] + pd.to_timedelta(final_df["shift"].map(shift_start_hour), unit="h")
hour_map = df_shift.groupby(["UCR_AGENCY_NAME", "shift_date", "shift"])["Date"].apply(lambda x: int(x.min().hour)).to_dict()
final_df["hour"] = final_df.apply(
    lambda row: hour_map.get((row["UCR_AGENCY_NAME"], row["shift_date"], row["shift"]), shift_start_hour[row["shift"]]),
    axis=1
)

print(f"Crime records used: {len(df_shift):,}")
print(f"Unique tiles: {df_shift['UCR_AGENCY_NAME'].nunique():,}")
print(f"Shift-level rows in final_df: {len(final_df):,}")

Crime records used: 1,272,576
Unique tiles: 620
Shift-level rows in final_df: 2,719,320


In [29]:
preview_cols = ["UCR_AGENCY_NAME", "shift_date", "shift", "crime_count", "target", "Date", "hour"]
final_df[preview_cols].head(10)

,UCR_AGENCY_NAME,shift_date,shift,crime_count,target,Date,hour
0,QUINCY,2020-12-31,morning_noon,0.0,0,2020-12-31 06:00:00,6
1,QUINCY,2020-12-31,afternoon_night,0.0,0,2020-12-31 14:00:00,14
2,QUINCY,2020-12-31,overnight,0.0,0,2020-12-31 22:00:00,22
3,QUINCY,2021-01-01,morning_noon,0.0,0,2021-01-01 06:00:00,6
4,QUINCY,2021-01-01,afternoon_night,0.0,0,2021-01-01 14:00:00,14
5,QUINCY,2021-01-01,overnight,0.0,0,2021-01-01 22:00:00,22
6,QUINCY,2021-01-02,morning_noon,0.0,0,2021-01-02 06:00:00,6
7,QUINCY,2021-01-02,afternoon_night,0.0,0,2021-01-02 14:00:00,14
8,QUINCY,2021-01-02,overnight,0.0,0,2021-01-02 22:00:00,22
9,QUINCY,2021-01-03,morning_noon,0.0,0,2021-01-03 06:00:00,6


# Feature Engineering

In [30]:
# Ensure Date is datetime and sort within each tile
final_df['Date'] = pd.to_datetime(final_df['Date'])
final_df = final_df.sort_values(['UCR_AGENCY_NAME', 'Date'])

In [31]:
final_df

,UCR_AGENCY_NAME,shift_date,shift,crime_count,target,Date,hour
1355274,ADAMS,2020-12-31,morning_noon,0.0,0,2020-12-31 06:00:00,6
1355275,ADAMS,2020-12-31,afternoon_night,0.0,0,2020-12-31 14:00:00,14
1355276,ADAMS,2020-12-31,overnight,0.0,0,2020-12-31 22:00:00,22
1355277,ADAMS,2021-01-01,morning_noon,0.0,0,2021-01-01 06:00:00,6
1355278,ADAMS,2021-01-01,afternoon_night,0.0,0,2021-01-01 14:00:00,14
...,...,...,...,...,...,...,...
1732465,ZION,2024-12-30,afternoon_night,1.0,1,2024-12-30 14:00:00,0
1732466,ZION,2024-12-30,overnight,0.0,0,2024-12-30 22:00:00,22
1732467,ZION,2024-12-31,morning_noon,0.0,0,2024-12-31 06:00:00,6
1732468,ZION,2024-12-31,afternoon_night,1.0,1,2024-12-31 14:00:00,0


In [33]:
# 1-Day Lag: What happened at this same shift yesterday?
# Group by tile AND shift so shift(1) compares same shift across days (true 1-day lag)
# e.g. morning_noon today <- morning_noon yesterday (not overnight 8h ago)
final_df['lag_1d'] = final_df.groupby(['UCR_AGENCY_NAME', 'shift'])['crime_count'].shift(1)

# Rolling Averages: The 'Momentum' of the tile
# Group by tile AND shift so window=7 means 7 calendar days (not 7 shifts ~2.3 days)
# use shift(1) to avoid data leakage (don't include current day in the average)
final_df['rolling_7d_mean'] = (
    final_df.groupby(['UCR_AGENCY_NAME', 'shift'])['crime_count']
    .transform(lambda x: x.shift(1).rolling(window=7, min_periods=1).mean())
)

final_df['rolling_30d_mean'] = (
    final_df.groupby(['UCR_AGENCY_NAME', 'shift'])['crime_count']
    .transform(lambda x: x.shift(1).rolling(window=30, min_periods=1).mean())
)

In [34]:
# Cyclical time features: Encode day of week and month as sine/cosine pairs to capture cyclical patterns without introducing artificial discontinuities
# Extract basic components
final_df['day_of_week'] = final_df['Date'].dt.dayofweek
final_df['month'] = final_df['Date'].dt.month

# Sine/Cosine Transformation for Day of Week (0-6)
# Sunday (6) is adjacent to Monday (0)
final_df['day_sin'] = np.sin(2 * np.pi * final_df['day_of_week'] / 7)
final_df['day_cos'] = np.cos(2 * np.pi * final_df['day_of_week'] / 7)

# Sine/Cosine Transformation for Month (1-12)
# Subtract 1 from month to make it 0-11 for the transformation, so December (12) is adjacent to January (1)
final_df['month_sin'] = np.sin(2 * np.pi * (final_df['month']-1) / 12)
final_df['month_cos'] = np.cos(2 * np.pi * (final_df['month']-1) / 12)

In [35]:
# Weekend flag
final_df['is_weekend'] = final_df['day_of_week'].isin([5, 6]).astype(int)

In [36]:
# Binary flag for late-night hours (22:00 to 01:59)
final_df['is_late_night'] = ((final_df['hour'] >= 22) | (final_df['hour'] < 2)).astype(int)

In [37]:
# Simple Seasonal Darkness Proxy 
# Chicago: Winter months (Nov, Dec, Jan, Feb) have more dark hours in Shift 1 (Afternoon)
final_df['is_winter_evening'] = (
    (final_df['month'].isin([11, 12, 1, 2])) & 
    (final_df['shift'] == 'afternoon_night')
).astype(int)

In [38]:
# Cyclical Shift encoding based on current shift labels
shift_to_idx = {'morning_noon': 0, 'afternoon_night': 1, 'overnight': 2}
shift_idx = final_df['shift'].map(shift_to_idx).fillna(0)

final_df['shift_sin'] = np.sin(2 * np.pi * shift_idx / 3)
final_df['shift_cos'] = np.cos(2 * np.pi * shift_idx / 3)

In [40]:
# Preview engineered features except Weekend flag (categorical flag is less interesting in a describe() summary)
feature_cols = ['UCR_AGENCY_NAME', 'lag_1d', 'rolling_7d_mean', 'rolling_30d_mean',
                'day_sin', 'day_cos', 'month_sin', 'month_cos', 
                'is_late_night', 'is_winter_evening', 'shift_sin', 'shift_cos']
print("\n--- Engineered Features Preview ---")
final_df[feature_cols].describe().round(2)


--- Engineered Features Preview ---


,lag_1d,rolling_7d_mean,rolling_30d_mean,day_sin,day_cos,month_sin,month_cos,is_late_night,is_winter_evening,shift_sin,shift_cos
count,2717460.00,2717460.00,2717460.00,2719320.00,2719320.00,2719320.00,2719320.00,2719320.00,2719320.00,2719320.00,2719320.00
mean,0.47,0.47,0.46,-0.00,0.00,-0.00,-0.00,0.43,0.11,0.00,-0.00
std,6.12,6.04,5.99,0.71,0.71,0.71,0.71,0.50,0.31,0.71,0.71
min,0.00,0.00,0.00,-0.97,-0.90,-1.00,-1.00,0.00,0.00,-0.87,-0.50
25%,0.00,0.00,0.00,-0.78,-0.90,-0.50,-0.87,0.00,0.00,-0.87,-0.50
50%,0.00,0.00,0.00,0.00,-0.22,0.00,-0.00,0.00,0.00,0.00,-0.50
75%,0.00,0.14,0.20,0.78,0.62,0.87,0.50,1.00,0.00,0.87,1.00
max,327.00,240.14,228.60,0.97,1.00,1.00,1.00,1.00,1.00,0.87,1.00


In [41]:
final_df

,UCR_AGENCY_NAME,shift_date,shift,crime_count,target,Date,hour,lag_1d,rolling_7d_mean,rolling_30d_mean,...,month,day_sin,day_cos,month_sin,month_cos,is_weekend,is_late_night,is_winter_evening,shift_sin,shift_cos
1355274,ADAMS,2020-12-31,morning_noon,0.0,0,2020-12-31 06:00:00,6,NaN,NaN,NaN,...,12,0.433884,-0.900969,-0.5,0.866025,0,0,0,0.000000,1.0
1355275,ADAMS,2020-12-31,afternoon_night,0.0,0,2020-12-31 14:00:00,14,NaN,NaN,NaN,...,12,0.433884,-0.900969,-0.5,0.866025,0,0,1,0.866025,-0.5
1355276,ADAMS,2020-12-31,overnight,0.0,0,2020-12-31 22:00:00,22,NaN,NaN,NaN,...,12,0.433884,-0.900969,-0.5,0.866025,0,1,0,-0.866025,-0.5
1355277,ADAMS,2021-01-01,morning_noon,0.0,0,2021-01-01 06:00:00,6,0.0,0.000000,0.000000,...,1,-0.433884,-0.900969,0.0,1.000000,0,0,0,0.000000,1.0
1355278,ADAMS,2021-01-01,afternoon_night,0.0,0,2021-01-01 14:00:00,14,0.0,0.000000,0.000000,...,1,-0.433884,-0.900969,0.0,1.000000,0,0,1,0.866025,-0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1732465,ZION,2024-12-30,afternoon_night,1.0,1,2024-12-30 14:00:00,0,1.0,0.428571,0.733333,...,12,0.000000,1.000000,-0.5,0.866025,0,1,1,0.866025,-0.5
1732466,ZION,2024-12-30,overnight,0.0,0,2024-12-30 22:00:00,22,0.0,0.285714,0.266667,...,12,0.000000,1.000000,-0.5,0.866025,0,1,0,-0.866025,-0.5
1732467,ZION,2024-12-31,morning_noon,0.0,0,2024-12-31 06:00:00,6,0.0,0.571429,0.233333,...,12,0.781831,0.623490,-0.5,0.866025,0,0,0,0.000000,1.0
1732468,ZION,2024-12-31,afternoon_night,1.0,1,2024-12-31 14:00:00,0,1.0,0.428571,0.733333,...,12,0.781831,0.623490,-0.5,0.866025,0,1,1,0.866025,-0.5


In [45]:
final_df.to_parquet('GeneralizationTest.parquet')